In [12]:
import torch

torch.cuda.is_available()

False

In [1]:
import os
from dotenv import load_dotenv
from huggingface_hub import login

load_dotenv()  # 이거 꼭 있어야됨

token = os.getenv("HF_TOKEN")
login(token)


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [2]:
import torch
import torch.nn as nn
from transformers import AutoModel
import inspect

model = AutoModel.from_pretrained(
    "facebook/dinov3-vits16plus-pretrain-lvd1689m"
)

print("=== model loaded ===")
print(model)


Loading weights:   0%|          | 0/235 [00:00<?, ?it/s]

=== model loaded ===
DINOv3ViTModel(
  (embeddings): DINOv3ViTEmbeddings(
    (patch_embeddings): Conv2d(3, 384, kernel_size=(16, 16), stride=(16, 16))
  )
  (rope_embeddings): DINOv3ViTRopePositionEmbedding()
  (layer): ModuleList(
    (0-11): 12 x DINOv3ViTLayer(
      (norm1): LayerNorm((384,), eps=1e-05, elementwise_affine=True)
      (attention): DINOv3ViTAttention(
        (k_proj): Linear(in_features=384, out_features=384, bias=False)
        (v_proj): Linear(in_features=384, out_features=384, bias=True)
        (q_proj): Linear(in_features=384, out_features=384, bias=True)
        (o_proj): Linear(in_features=384, out_features=384, bias=True)
      )
      (layer_scale1): DINOv3ViTLayerScale()
      (drop_path): Identity()
      (norm2): LayerNorm((384,), eps=1e-05, elementwise_affine=True)
      (mlp): DINOv3ViTGatedMLP(
        (gate_proj): Linear(in_features=384, out_features=1536, bias=True)
        (up_proj): Linear(in_features=384, out_features=1536, bias=True)
        (d

In [3]:
model.layer

ModuleList(
  (0-11): 12 x DINOv3ViTLayer(
    (norm1): LayerNorm((384,), eps=1e-05, elementwise_affine=True)
    (attention): DINOv3ViTAttention(
      (k_proj): Linear(in_features=384, out_features=384, bias=False)
      (v_proj): Linear(in_features=384, out_features=384, bias=True)
      (q_proj): Linear(in_features=384, out_features=384, bias=True)
      (o_proj): Linear(in_features=384, out_features=384, bias=True)
    )
    (layer_scale1): DINOv3ViTLayerScale()
    (drop_path): Identity()
    (norm2): LayerNorm((384,), eps=1e-05, elementwise_affine=True)
    (mlp): DINOv3ViTGatedMLP(
      (gate_proj): Linear(in_features=384, out_features=1536, bias=True)
      (up_proj): Linear(in_features=384, out_features=1536, bias=True)
      (down_proj): Linear(in_features=1536, out_features=384, bias=True)
      (act_fn): SiLUActivation()
    )
    (layer_scale2): DINOv3ViTLayerScale()
  )
)

In [4]:
from src.re_make_Gate import inject_gating, freeze_except_gate

In [5]:
for i in model.parameters():
    i.required_grad=True

In [6]:
model, hooks = inject_gating(model, gate_type="elementwise",keep_cls_ungated=True,target_layers=None)

[inject_gating] 총 12개 중 12개 레이어에 G1 gate 적용
  → 적용 인덱스: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11]  (gate_type=elementwise)


In [7]:
from src.model import dinosplus_classfier
vit=dinosplus_classfier(model,200)


In [8]:
print(vit)

dinosplus_classfier(
  (backbone): DINOv3ViTModel(
    (embeddings): DINOv3ViTEmbeddings(
      (patch_embeddings): Conv2d(3, 384, kernel_size=(16, 16), stride=(16, 16))
    )
    (rope_embeddings): DINOv3ViTRopePositionEmbedding()
    (layer): ModuleList(
      (0-11): 12 x DINOv3ViTLayer(
        (norm1): LayerNorm((384,), eps=1e-05, elementwise_affine=True)
        (attention): DINOv3ViTAttention(
          (k_proj): Linear(in_features=384, out_features=384, bias=False)
          (v_proj): Linear(in_features=384, out_features=384, bias=True)
          (q_proj): Linear(in_features=384, out_features=384, bias=True)
          (o_proj): GatedOutputProjection(
            (original_o_proj): Linear(in_features=384, out_features=384, bias=True)
            (gate_proj): Linear(in_features=384, out_features=384, bias=True)
          )
        )
        (layer_scale1): DINOv3ViTLayerScale()
        (drop_path): Identity()
        (norm2): LayerNorm((384,), eps=1e-05, elementwise_affine=True)


In [9]:
from datasets import load_dataset
from src.data import setdata
from torch.utils.data import DataLoader
data=load_dataset('zh-plus/tiny-imagenet')
train=setdata(data['train'])
test=setdata(data['valid'])
train_loader=DataLoader(train,batch_size=128,pin_memory=True,num_workers=4,shuffle=True)
test_loader=DataLoader(test,batch_size=128,pin_memory=True,num_workers=4,shuffle=True)

In [10]:
import torch.nn as nn
import torch.optim as optim
import torch
optimizer=optim.Adam(vit.parameters(),lr=0.0001)
criterion=nn.CrossEntropyLoss()
scaler=torch.amp.GradScaler()

/tmp/ipykernel_7049/1847518477.py:6: UserWarning: torch.cuda.amp.GradScaler is enabled, but CUDA is not available.  Disabling.
  scaler=torch.amp.GradScaler()


In [11]:
import torch
print(torch.cuda.get_device_name(0))
device= 'cuda' if torch.cuda.is_available() else 'cpu'

RuntimeError: Found no NVIDIA driver on your system. Please check that you have an NVIDIA GPU and installed a driver from http://www.nvidia.com/Download/index.aspx

In [ ]:
import torch

torch.backends.cuda.enable_flash_sdp(True)
torch.backends.cuda.enable_mem_efficient_sdp(True)
torch.backends.cuda.enable_math_sdp(False)

In [ ]:
from src.training import train 
train(vit,train_loader,test_loader,50,criterion,scaler,device,optimizer,'full fine tuning vit dino v3 splus gated last ONLY','/home/hyuksu/deep-learning-study/outputs/best dino v3 splus gated last ONLY.pth')